## preprocessing markdown file into Milvus DB

### 1. spin up milvusDB ( in ubuntu OS ) using the script below : 
```bash standalone_milvus.sh
#!/usr/bin/env bash

# Licensed to the LF AI & Data foundation under one
# or more contributor license agreements. See the NOTICE file
# distributed with this work for additional information
# regarding copyright ownership. The ASF licenses this file
# to you under the Apache License, Version 2.0 (the
# "License"); you may not use this file except in compliance
# with the License. You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

run_embed() {
    cat << EOF > embedEtcd.yaml
listen-client-urls: http://0.0.0.0:2379
advertise-client-urls: http://0.0.0.0:2379
quota-backend-bytes: 4294967296
auto-compaction-mode: revision
auto-compaction-retention: '1000'
EOF

    cat << EOF > user.yaml
# Extra config to override default milvus.yaml
EOF

    sudo docker run -d \
        --name milvus-standalone \
        --security-opt seccomp:unconfined \
        -e ETCD_USE_EMBED=true \
        -e ETCD_DATA_DIR=/var/lib/milvus/etcd \
        -e ETCD_CONFIG_PATH=/milvus/configs/embedEtcd.yaml \
        -e COMMON_STORAGETYPE=local \
        -v $(pwd)/volumes/milvus:/var/lib/milvus \
        -v $(pwd)/embedEtcd.yaml:/milvus/configs/embedEtcd.yaml \
        -v $(pwd)/user.yaml:/milvus/configs/user.yaml \
        -p 19530:19530 \
        -p 9091:9091 \
        -p 2379:2379 \
        --health-cmd="curl -f http://localhost:9091/healthz" \
        --health-interval=30s \
        --health-start-period=90s \
        --health-timeout=20s \
        --health-retries=3 \
        milvusdb/milvus:v2.4.5 \
        milvus run standalone  1> /dev/null
}

wait_for_milvus_running() {
    echo "Wait for Milvus Starting..."
    while true
    do
        res=`sudo docker ps|grep milvus-standalone|grep healthy|wc -l`
        if [ $res -eq 1 ]
        then
            echo "Start successfully."
            echo "To change the default Milvus configuration, add your settings to the user.yaml file and then restart the service."
            break
        fi
        sleep 1
    done
}

start() {
    res=`sudo docker ps|grep milvus-standalone|grep healthy|wc -l`
    if [ $res -eq 1 ]
    then
        echo "Milvus is running."
        exit 0
    fi

    res=`sudo docker ps -a|grep milvus-standalone|wc -l`
    if [ $res -eq 1 ]
    then
        sudo docker start milvus-standalone 1> /dev/null
    else
        run_embed
    fi

    if [ $? -ne 0 ]
    then
        echo "Start failed."
        exit 1
    fi

    wait_for_milvus_running
}

stop() {
    sudo docker stop milvus-standalone 1> /dev/null

    if [ $? -ne 0 ]
    then
        echo "Stop failed."
        exit 1
    fi
    echo "Stop successfully."

}

delete() {
    res=`sudo docker ps|grep milvus-standalone|wc -l`
    if [ $res -eq 1 ]
    then
        echo "Please stop Milvus service before delete."
        exit 1
    fi
    sudo docker rm milvus-standalone 1> /dev/null
    if [ $? -ne 0 ]
    then
        echo "Delete failed."
        exit 1
    fi
    sudo rm -rf $(pwd)/volumes
    sudo rm -rf $(pwd)/embedEtcd.yaml
    sudo rm -rf $(pwd)/user.yaml
    echo "Delete successfully."
}


case $1 in
    restart)
        stop
        start
        ;;
    start)
        start
        ;;
    stop)
        stop
        ;;
    delete)
        delete
        ;;
    *)
        echo "please use bash standalone_embed.sh restart|start|stop|delete"
        ;;
esac
```


### 2. spin up milvusDB server calling bash standalone_milvue.sh start
### 3. Go through the below and process the xxx.md file ingesting into milvusDB

    

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_nvidia_ai_endpoints import ChatNVIDIA , NVIDIAEmbeddings
#from langchain.vectorstores import Chroma
from langchain.vectorstores import FAISS
from langchain_text_splitters import MarkdownHeaderTextSplitter
import uuid
import argparse
from langchain_text_splitters import CharacterTextSplitter

## process the PDF AutoDesk Genesis Brochure
def process_markdown_file(md_file):
    f=open(md_file, "r", encoding="utf-8")
    markdown_document = '\n'.join(f.readlines())
    
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
    ]
    ## create documents with Langchain's MarkdownHeaderTextSplitter 
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    docs_and_metadata = markdown_splitter.split_text(markdown_document)
    return docs_and_metadata


In [22]:
files=os.listdir('./data_dir/')
source_ref=[]
documents=[]
for file in files:
    if file.endswith('.md'):
        source_name=file
        
        docs=process_markdown_file('./data_dir/'+file)
        n=len(docs)
        documents.append(docs)
        source_ref.append([source_name for _ in range(n)])
        

In [23]:
len(documents), len(source_ref)

(1, 1)

In [27]:
from itertools import chain
documents_flatten=list(chain(*documents))
source_ref_flatten=list(chain(*source_ref))
len(documents_flatten), len(source_ref_flatten)

(6, 6)

In [7]:
## ensure transformers and sentence-transformers versions compatibility for the Qwen3 embedding
!pip install transformers>=4.51.0
!pip install sentence-transformers>=2.7.0

In [9]:
from sentence_transformers import SentenceTransformer
# supported langauge : https://github.com/QwenLM/Qwen3-Embedding
# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )

# The queries and documents to embed
queries = [
    "förklarar svenskautbildningssystemet för mig, tack",
    "Är förskoleklass obligatorisk i Sverige och när börjar den?",
]
documents = [doc.page_content for doc in docs]

# Encode the queries and documents. Note that queries benefit from using a prompt
# Here we use the prompt called "query" stored under `model.prompts`, but you can
# also pass your own prompt via the `prompt` argument
query_embeddings = model.encode(queries, prompt_name="query")
document_embeddings = model.encode(documents)

# Compute the (cosine) similarity between the query and document embeddings
similarity = model.similarity(query_embeddings, document_embeddings)
print(similarity)
# tensor([[0.7646, 0.1414],
#         [0.1355, 0.6000]])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

tensor([[0.6314, 0.2670, 0.3328, 0.2379, 0.2658, 0.6140],
        [0.6001, 0.3697, 0.3138, 0.2306, 0.2145, 0.4220]])


In [5]:
## reference source url : https://milvus.io/docs/milvus_rag_with_vllm.md
## class reference source url : https://discuss.huggingface.co/t/text-input-bigger-than-max-tokens-length-for-semantic-search-embeddings/64465/2
import numpy as np
from sentence_transformers import SentenceTransformer
class Qwen3EmbeddingModel:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")            
            cls._instance.max_tokens = 1024 # Your model's max tokens limit
            cls._instance.overlap = 0 # no of tokens to be overlapped between chunks
        return cls._instance
    
    def get_embeddings(self, docs, prompt_name=None):   
        isquery= "embed a query" if prompt_name!=None else "embed documents"
        print(isquery)
        return self.model.encode(docs, prompt_name=prompt_name)
    

In [6]:
embedder=Qwen3EmbeddingModel()

In [7]:
## manually create embedding for each document 
collect_embedding_vectors=[]
documents = [doc.page_content for doc in docs]

for doc in docs:
    embed_vector=embedder.get_embeddings(doc.page_content)
    #type(embed_vector), embed_vector.shape
    collect_embedding_vectors.append(embed_vector)
len(collect_embedding_vectors)

embed documents
embed documents
embed documents
embed documents
embed documents
embed documents


6

In [8]:
### prepare for feeding to milvus vectorDB

# Create dict_list for Milvus insertion.
dict_list = []
n=len(docs)
id_int64=[int(i) for i in range(n)]
for doc, vector , id in zip(docs, collect_embedding_vectors, id_int64 ):
   # Assemble embedding vector, original text chunk, metadata.
   chunk_dict = {
       'id':np.int64(id),
       'chunk': doc.page_content,
       'source': doc.metadata.get('source', ""),
       'vector': np.float32(vector), # Milvus expect the vector to be of type np.float32
   }
   dict_list.append(chunk_dict)



## populating MilvusDB 

In [ ]:
## https://milvus.io/docs/quickstart.md
!pip install "pymilvus[model]"
!pip install langchain-milvus

In [9]:
## following Milvus documentation https://milvus.io/docs/insert-update-delete.md
from pymilvus import MilvusClient
import time
client = MilvusClient(
    uri="http://localhost:19530"
)
COLLECTION_NAME = "SovAISwedish"
EMBEDDING_DIM=1024

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIM,
    metric_type="IP"
)



print("Start inserting entities")
start_time = time.time()
res = client.insert(
    collection_name=COLLECTION_NAME,
    data=dict_list,
    progress_bar=True
)

print(res)

end_time = time.time()
print(f"Milvus insert time for {len(dict_list)} vectors: ", end="")
print(f"{round(end_time - start_time, 2)} seconds")



Start inserting entities
{'insert_count': 6, 'ids': [0, 1, 2, 3, 4, 5]}
Milvus insert time for 6 vectors: 0.06 seconds


In [10]:
SAMPLE_QUESTION = "Är förskoleklass obligatorisk i Sverige och när börjar den?"


query_embeddings = embedder.get_embeddings(SAMPLE_QUESTION, prompt_name="query")
query_embeddings = np.float32(query_embeddings)


embed a query


In [11]:
client.list_collections()


['SovAISwedish']

In [12]:


OUTPUT_FIELDS = list(dict_list[0].keys())
OUTPUT_FIELDS.remove('vector')
print(OUTPUT_FIELDS)

search_params = {"metric_type": "IP","params":{"nprobe":1024}}


TOP_K = 2

results = client.search(
    COLLECTION_NAME,
    data=[query_embeddings],
    limit=TOP_K,
    search_params=search_params,
    output_fields=OUTPUT_FIELDS,
    consistency_level="Eventually")


['id', 'chunk', 'source']


In [13]:
results[0]

[{'id': 0, 'distance': 0.6001015901565552, 'entity': {'id': 0, 'chunk': 'Det svenska utbildningssystemet är uppbyggt i flera steg och inkluderar obligatoriska och  \nfrivilliga utbildningar. Den grundläggande utbildningen omfattar förskoleklass och grundskola,  \nsom är obligatoriska för alla barn och ungdomar. Efter grundskolan finns det ett valfritt  \ngymnasium, följt av högre studier på universitet, högskola eller yrkeshögskola.  \nUtbildningssystemet i korthet:  \n- Förskola: Rätt att gå för barn mellan 1 och 5 år.  \n- Förskoleklass: Obligatorisk från det år barnet fyller sex år.  \n- Grundskola: Obligatorisk och omfattar 9 årskurser.  \n- Gymnasium: Frivillig utbildning efter grundskolan, som bereder för högre studier.  \n- Högskola/Universitet/Yrkeshögskola: Högre utbildning för vuxna.  \n- Sfi (Svenska för invandrare): Utbildning för invandrare som vill lära sig svenska.  \n- Komvux (Kommunal vuxenutbildning): Utbildning för vuxna på grundläggande och  \n- gymnasial nivå.', 's

## user RBAC control https://milvus.io/docs/v2.2.x/rbac.md